<a href="https://colab.research.google.com/github/shuangquan-li-con/Econ5220-Computation-Finance-Financial-Econometrics/blob/main/HW6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from itertools import combinations

file_path = "/content/25eq2021 fac tst.xlsx"
adj = pd.read_excel(file_path, sheet_name="adj pr")
rtn = pd.read_excel(file_path, sheet_name="rtn")

stocks = ["PFE", "MDT", "ABT", "TMO", "MRK"]
logp = np.log(adj[stocks])

In [2]:
def df_tau_mu(series):
    y = series.dropna()
    dy = y.diff()
    y_lag = y.shift(1)
    reg = pd.concat([dy, y_lag], axis=1).dropna()
    reg.columns = ["dy", "y_lag"]
    X = sm.add_constant(reg["y_lag"])
    model = sm.OLS(reg["dy"], X).fit()
    return model.tvalues["y_lag"]

for s in stocks:
    print(s, df_tau_mu(logp[s]))

PFE -0.3978406764547777
MDT 0.02048142253229399
ABT 0.6603687078920731
TMO 0.5787469438162446
MRK -1.0062301474073083


In [3]:
def engle_granger_manual(y, x):
    tmp = pd.concat([y, x], axis=1).dropna()
    tmp.columns = ["y", "x"]
    X = sm.add_constant(tmp["x"])
    step1 = sm.OLS(tmp["y"], X).fit()
    e = step1.resid
    de = e.diff()
    e_lag = e.shift(1)
    reg = pd.concat([de, e_lag], axis=1).dropna()
    reg.columns = ["de", "e_lag"]
    step2 = sm.OLS(reg["de"], reg[["e_lag"]]).fit()
    return step2.tvalues["e_lag"], e.iloc[-1], e.std(ddof=1)

for a, b in combinations(stocks, 2):
    stat, last_resid, resid_sd = engle_granger_manual(logp[a], logp[b])
    print(a, b, stat, last_resid, resid_sd)

PFE MDT -3.1150242971339575 -0.10632424649198802 0.14964012249739397
PFE ABT -1.9891310029053046 -0.15993434393069839 0.1675350564764562
PFE TMO -2.403160689015343 -0.14517188907140488 0.16488197553648537
PFE MRK -3.2466977879801067 0.03734711174539607 0.14114251278040602
MDT ABT -2.423424152675978 0.007522412377690912 0.18200646620626384
MDT TMO -2.790535945404901 0.009621084569233496 0.16463256305552704
MDT MRK -3.0196562971573324 0.21993799915701917 0.1840552035280412
ABT TMO -3.437697170440472 0.04700170920698987 0.11506358391134193
ABT MRK -2.532681656270511 0.35472433551520144 0.18581034004580563
TMO MRK -3.165017771444222 0.4215362661938409 0.2071710779954385


In [4]:
!pip install arch
from arch import arch_model

pfe_ret = rtn["PFE"].dropna()
garch = arch_model(pfe_ret, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")
res = garch.fit(disp="off")
print(res.summary())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 13.3 MB/s eta 0:00:00
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                    PFE   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                292.188
Distribution:                  Normal   AIC:                          -576.376
Method:            Maximum Likelihood   BIC:                          -563.243
                                        No. Observations:                  197
Date:                Fri, Mar 20 2026   Df Residuals:                      196
Time:                        20:54:46   Df Model:                            1
                                 Mean Model                                 
                 coef    std err          t      P>|t|      95.0% Conf. Int.
------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.003234. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
